# Popular Data Science Questions

We work for a company that creates data science content (books, online articles, videos, interactive learning platforms). Our goal is to use [Data Science Stack Exchange](https://datascience.stackexchange.com) (DSSE) to figure out **what content we should create** — that is, which data science topics people are most interested in learning about.

## Stack Exchange

Stack Exchange hosts question-and-answer sites on many topics. Each site is focused on one subject, and DSSE is dedicated to data science.

- **What kind of questions are welcome?** Practical, answerable questions about data science, machine learning, statistics and related tools. Questions that are too broad, opinion-based or not about data science get closed.
- **What does the home page subdivide into?** Besides *Questions*, the site has *Tags* (topic labels), *Users*, *Unanswered* questions and *Companies*. The **Tags** section is especially useful for us: every question is labeled with up to five tags that describe its subject.
- **What information is available in each post?** The title and body, tags, score (upvotes minus downvotes), number of views, answers, comments and favorites, the creation date, and the author.

Tags give us a convenient way to classify questions by topic, and the view count tells us how popular each question is.

## Stack Exchange Data Explorer

The [Stack Exchange Data Explorer](https://data.stackexchange.com/datascience/query/new) (SEDE) lets us query DSSE's database with SQL. The most promising tables for our goal are:

- **Posts** — one row per post (questions *and* answers), with columns such as `PostTypeId`, `CreationDate`, `Score`, `ViewCount`, `Tags`, `AnswerCount` and `FavoriteCount`;
- **Tags** — each tag with its total `Count` of posts;
- **PostTags** — which tags belong to which post;
- **PostTypes** — the meaning of each `PostTypeId` (1 = question, 2 = answer).

The Posts table contains everything we need, so we'll focus on it.

## Getting the Data

We ran the following query in SEDE to get all the questions asked in 2019 and saved the result as `2019_questions.csv`:

```sql
SELECT Id, CreationDate,
       Score, ViewCount, Tags,
       AnswerCount, FavoriteCount
  FROM posts
 WHERE PostTypeId = 1 AND YEAR(CreationDate) = 2019;
```

## Exploring the Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

questions = pd.read_csv("2019_questions.csv", parse_dates=["CreationDate"])
print(questions.shape)
questions.head()

In [ ]:
questions.info()

In [ ]:
questions.isnull().sum()

The dataset has **8,839 questions** from 2019 and 7 columns.

- **Missing values:** only `FavoriteCount` has missing values — 7,432 of them (only 1,407 rows are filled). A missing value most likely means the question was never marked as a favorite, so we can safely replace it with `0`.
- **Types:** `FavoriteCount` is a float only because of the missing values; after filling them it should be an integer. `CreationDate` was already parsed as a datetime, and the other numeric columns are integers.
- **Tags:** each value is a string like `<python><keras><cnn>`. To count tags, we'll turn it into a Python list (`['python', 'keras', 'cnn']`).

## Cleaning the Data

In [ ]:
questions["FavoriteCount"] = questions["FavoriteCount"].fillna(0).astype(int)
questions["Tags"] = questions["Tags"].str.replace("^<|>$", "", regex=True).str.split("><")
print(questions.dtypes)
questions.sample(3, random_state=1)

## Most Used and Most Viewed Tags

In [ ]:
tag_count = questions["Tags"].explode().value_counts()
most_used = tag_count.head(20).sort_values()
print(tag_count.head(10))

most_used.plot(kind="barh", figsize=(10, 8), title="20 most used tags (2019)")
plt.show()

In [ ]:
tag_view_count = questions.explode("Tags").groupby("Tags")["ViewCount"].sum().sort_values(ascending=False)
most_viewed = tag_view_count.head(20).sort_values()
print(tag_view_count.head(10))

most_viewed.plot(kind="barh", figsize=(10, 8), title="20 most viewed tags (2019)")
plt.show()

In [ ]:
in_used = set(most_used.index)
in_viewed = set(most_viewed.index)
print("Only in most used:", sorted(in_used - in_viewed))
print("Only in most viewed:", sorted(in_viewed - in_used))

**Most used tags (number of questions):** `machine-learning` (2,693), `python` (1,814), `deep-learning` (1,220), `neural-network` (1,055), `keras` (935), `classification` (685), `tensorflow` (584), `scikit-learn` (540), `nlp` (493) and `cnn` (489).

**Most viewed tags (total views):** `python` (537,585), `machine-learning` (388,499), `keras` (268,608), `deep-learning` (233,628), `pandas` (201,787), `neural-network` (185,367), `scikit-learn`, `tensorflow`, `classification` and `dataframe`.

The two lists overlap a lot: 16 of the 20 tags appear in both. Tags such as `r`, `statistics` and `predictive-modeling` are used often but viewed less, while `pandas`-related tags (`dataframe`, `csv`, `numpy`) and `pytorch` are viewed a lot without being among the most used. Several of the top tags are deep learning related: `deep-learning`, `neural-network`, `keras`, `tensorflow` and `cnn`.

## Relations Between Tags

One way to find relationships between **pairs** of tags is to count how many times each pair appears together in the same question. We can store these counts in a matrix with one row and one column per tag. Another option is to compare the co-occurrence with what we'd expect if tags were independent (for example with a correlation or a lift measure), so that very popular tags like `machine-learning` don't dominate every pair.

For relationships between **several** tags at once, we could use association rule mining (e.g. the apriori algorithm), which finds groups of tags that frequently appear together, or cluster questions by their tag sets.

Below we compute the pair counts for the 20 most used tags.

In [ ]:
top_tags = list(tag_count.head(20).index)
associations = pd.DataFrame(0, index=top_tags, columns=top_tags)

for tags in questions["Tags"]:
    present = [tag for tag in tags if tag in top_tags]
    for t1 in present:
        for t2 in present:
            associations.loc[t1, t2] += 1

for tag in top_tags:
    associations.loc[tag, tag] = 0

associations.loc[["machine-learning", "deep-learning", "keras", "tensorflow", "neural-network"],
                 ["python", "machine-learning", "deep-learning", "keras", "tensorflow", "neural-network"]]

Among the most used tags, the deep learning tags appear together very often. For example, `deep-learning` appears with `machine-learning` in 429 questions, with `neural-network` in 305 and with `keras` in 247; `keras` and `tensorflow` appear together in 256 questions. `python` is mostly paired with `keras` (280) and `tensorflow` (167).

These pair counts are only a first step: very common tags such as `machine-learning` naturally co-occur with almost everything. Still, the strong links between `deep-learning`, `neural-network`, `keras` and `tensorflow` suggest that these tags describe one common topic.

## Enter Domain Knowledge

Many of the most popular tags are closely related: `keras` and `tensorflow` are deep learning libraries, and `neural-network`, `cnn`, `lstm` and `deep-learning` are deep learning concepts. `scikit-learn` and `pandas` are Python libraries, while `classification`, `regression` and `clustering` are machine learning tasks. Looking at the most popular tags with this knowledge, a strong candidate topic emerges: **deep learning**.

Before recommending it, we should check that this interest isn't just a fad. To do that, we'll track how the share of deep learning questions changed over time using all the questions ever asked on DSSE.

## Is Deep Learning Just a Fad?

In [ ]:
all_q = pd.read_csv("all_questions.csv", parse_dates=["CreationDate"])
all_q["Tags"] = all_q["Tags"].str.replace("^<|>$", "", regex=True).str.split("><")
print(all_q.shape)
all_q.head()

We classify a question as a **deep learning** question if it has at least one of the following tags: `lstm`, `cnn`, `scikit-learn`, `tensorflow`, `keras`, `neural-network`, `deep-learning`. (`scikit-learn` isn't strictly a deep learning library, but it appears together with these tags very often, so we keep it to follow the same definition throughout.)

In [ ]:
dl_tags = {"lstm", "cnn", "scikit-learn", "tensorflow", "keras", "neural-network", "deep-learning"}
all_q["DeepLearning"] = all_q["Tags"].apply(lambda tags: int(any(tag in dl_tags for tag in tags)))

# Keep only complete quarters (drop the most recent, incomplete quarter)
all_q["Quarter"] = all_q["CreationDate"].dt.to_period("Q")
last_quarter = all_q["Quarter"].max()
all_q = all_q[all_q["Quarter"] < last_quarter]

quarterly = all_q.groupby("Quarter").agg(DeepLearningQuestions=("DeepLearning", "sum"),
                                         TotalQuestions=("DeepLearning", "size"))
quarterly["DeepLearningRate"] = quarterly["DeepLearningQuestions"] / quarterly["TotalQuestions"]
quarterly

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
quarterly["TotalQuestions"].plot(kind="bar", ax=ax1, color="lightgray", alpha=0.7, label="Total questions")
ax1.set_ylabel("Questions per quarter")

ax2 = ax1.twinx()
ax2.plot(range(len(quarterly)), quarterly["DeepLearningRate"], color="tab:blue", marker="o", label="Deep learning share")
ax2.set_ylabel("Share of deep learning questions")

ax1.set_xticklabels([str(q) for q in quarterly.index], rotation=45)
ax1.set_title("Deep learning questions on DSSE per quarter")
fig.legend(loc="upper left", bbox_to_anchor=(0.1, 0.9))
plt.tight_layout()
plt.show()

The share of deep learning questions grew steadily from **about 6% in 2014 Q2** to **about 35% in 2017** and peaked at **45% in 2018 Q1 and Q3**. In 2019 it stabilized at around **39–40%** of all questions.

At the same time, the total number of questions kept increasing (from 157 in 2014 Q2 to more than 2,000 per quarter in 2019), so the absolute number of deep learning questions also kept growing — from 9 to 800–940 per quarter. Interest in deep learning doesn't look like a fad: it grew quickly and then settled at a high, stable level.

## Conclusion

- The most popular topics on DSSE in 2019, both by number of questions and by views, are **machine learning, Python and deep learning** (`deep-learning`, `neural-network`, `keras`, `tensorflow`, `cnn`).
- Deep learning tags frequently appear together, which confirms that they form one topic.
- Over time, deep learning questions grew from about 6% to about 40% of all questions and the share has stayed high and stable since 2018, so the interest isn't slowing down.

**Recommendation:** create deep learning content first, ideally hands-on content in Python with Keras/TensorFlow. Since `pandas` and data-manipulation tags are among the most viewed, content that combines data preparation with deep learning would likely be popular too. We should keep tracking the trend over the coming quarters to confirm that interest remains stable.